# Day 2 — Step 2: 전처리 [필수 과제 문제1]

01_data_collection에서 저장한 데이터를 불러와  
가격 관련 파생 지표 3가지를 계산합니다.

---

## 이번 단계에서 추가할 컬럼

| 컬럼 | 계산식 | 의미 |
|------|--------|------|
| `price_change` | `close - open` | 종가 - 시가 (원 단위 변동) |
| `price_change_pct` | `price_change / open × 100` | 시가 대비 변동률 (%) |
| `high_low_diff` | `high - low` | 고가 - 저가 (일중 변동폭) |

---

### 💡 각 지표의 의미

**`price_change` (가격 변동)**
```
양수(+) : 종가 > 시가 → 상승 마감 (양봉)
음수(-) : 종가 < 시가 → 하락 마감 (음봉)
```

**`price_change_pct` (변동률)**
```
금액이 아닌 비율(%)로 비교 → 가격대가 다른 종목 간 공정한 비교 가능
예) BTC +200만원 vs SOL +200만원 → 금액은 같지만 비율은 다름
```

**`high_low_diff` (일중 변동폭)**
```
값이 크다 → 하루에 가격이 많이 움직였다 → 변동성(위험)이 크다
값이 작다 → 가격이 안정적 → 변동성이 작다
```

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

In [ ]:
# ── 01단계에서 저장한 데이터 불러오기 ────────────────────────────
# pd.read_csv() : CSV 파일을 DataFrame으로 읽기
# parse_dates=['date'] : 'date' 컬럼을 문자열이 아닌 datetime 타입으로 읽기
df = pd.read_csv('ohlcv_raw.csv', parse_dates=['date'])

print(f"데이터 로드 완료: {len(df)}행")
print(df.head())

---
## [필수 과제 문제1] 종가와 시가의 차이 계산

In [ ]:
def add_price_features(df):
    """
    가격 관련 파생 컬럼 3개를 계산하여 DataFrame에 추가합니다.

    추가되는 컬럼:
        price_change     : 종가 - 시가
        price_change_pct : 시가 대비 변동률 (%)
        high_low_diff    : 고가 - 저가 (일중 변동폭)

    매개변수:
        df : open, high, low, close 컬럼이 있는 DataFrame

    반환값:
        파생 컬럼이 추가된 DataFrame
    """
    # .copy() : 원본 df를 건드리지 않도록 복사본에 작업
    df = df.copy()

    # ── 컬럼1: price_change ───────────────────────────────────────
    # DataFrame의 컬럼 간 계산은 같은 행끼리 자동으로 맞춰서 계산됨 (벡터 연산)
    df['price_change'] = df['close'] - df['open']

    # ── 컬럼2: price_change_pct ───────────────────────────────────
    # price_change를 시가(open)로 나눠 퍼센트로 표현
    # 예) +2,000,000 / 90,000,000 × 100 ≈ +2.22%
    df['price_change_pct'] = (df['price_change'] / df['open']) * 100

    # ── 컬럼3: high_low_diff ──────────────────────────────────────
    # 하루 안에서 가격이 최대 얼마나 움직였는지
    df['high_low_diff'] = df['high'] - df['low']

    return df

In [ ]:
# ── 함수 적용 ──────────────────────────────────────────────────────
df = add_price_features(df)

print("[문제1 결과 확인]")
check_cols = ['ticker', 'date', 'open', 'close', 'price_change', 'price_change_pct', 'high_low_diff']
print(df[check_cols].head(10).to_string(index=False))

In [ ]:
# ── 종목별 평균 변동률 확인 ───────────────────────────────────────
# groupby('ticker') : 같은 종목끼리 묶어서 평균 계산
print("[종목별 평균 가격 변동률 (%)]") 
print(df.groupby('ticker')[['price_change_pct', 'high_low_diff']].mean().round(2))

In [ ]:
# ── 다음 단계(03_moving_average)로 전달 ──────────────────────────
df.to_csv('ohlcv_preprocessed.csv', index=False)
print("ohlcv_preprocessed.csv 저장 완료 → 03_moving_average.ipynb에서 불러옵니다")